# 🧠 Task 5: Mental Health Support Chatbot (Fine-Tuned)
**DevelopersHub Corporation — AI/ML Engineering Internship**

**Objective:** Fine-tune GPT-Neo 125M on EmpatheticDialogues to build an empathetic chatbot for stress, anxiety & emotional wellness.

---
**Model:** EleutherAI/GPT-Neo-125M | **Dataset:** EmpatheticDialogues (Facebook AI) | **Interface:** Colab interactive chat

> ⚠️ **Enable GPU:** Runtime → Change runtime type → T4 GPU

## Step 1: Install Required Libraries

In [ ]:
!pip install transformers datasets accelerate -q

## Step 2: Import Libraries

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import math
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from torch.utils.data import Dataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Using device: {device}")
if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

## Step 3: Load & Explore EmpatheticDialogues Dataset

In [ ]:
print("📥 Loading EmpatheticDialogues dataset...")
dataset = load_dataset("bdotloh/empathetic-dialogues-contexts")
print(dataset)
print("\n📝 Sample Entry:")
for k, v in dataset['train'][0].items():
    print(f"  {k}: {v}")

In [ ]:
train_df = pd.DataFrame(dataset['train'])
print(f"Shape: {train_df.shape}")
print(f"Columns: {list(train_df.columns)}")

# Auto-detect column names
emotion_col = next((c for c in train_df.columns if any(x in c.lower() for x in ['emotion','context','label'])), train_df.columns[0])
text_col    = next((c for c in train_df.columns if any(x in c.lower() for x in ['utterance','text','response','dialogue'])), train_df.columns[1])
prompt_col  = next((c for c in train_df.columns if 'prompt' in c.lower()), None)

print(f"\n🎭 Emotion column : '{emotion_col}'")
print(f"💬 Text column    : '{text_col}'")
print(f"📝 Prompt column  : '{prompt_col}'")
print(f"\nTop 15 emotions:\n{train_df[emotion_col].value_counts().head(15)}")

import builtins
builtins.EMOTION_COL = emotion_col
builtins.TEXT_COL    = text_col
builtins.PROMPT_COL  = prompt_col

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('EmpatheticDialogues — Dataset Overview', fontsize=14, fontweight='bold')

emotion_counts = train_df[EMOTION_COL].value_counts().head(15)
colors = plt.cm.RdYlGn([i/15 for i in range(15)])
axes[0].barh(emotion_counts.index[::-1], emotion_counts.values[::-1], color=colors)
axes[0].set_title('Top 15 Emotion Categories')
axes[0].set_xlabel('Count')
for i, v in enumerate(emotion_counts.values[::-1]):
    axes[0].text(v+5, i, str(v), va='center', fontsize=8)

train_df['text_len'] = train_df[TEXT_COL].astype(str).str.len()
axes[1].hist(train_df['text_len'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Text Length Distribution')
axes[1].set_xlabel('Character Length')
axes[1].set_ylabel('Frequency')
axes[1].axvline(train_df['text_len'].mean(), color='red', linestyle='--',
                label=f"Mean: {train_df['text_len'].mean():.0f}")
axes[1].legend()

plt.tight_layout()
plt.savefig('emotion_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Plot saved!")

## Step 4: Preprocess & Prepare Training Data

In [ ]:
mental_health_emotions = [
    'anxious','afraid','sad','lonely','devastated',
    'terrified','angry','furious','ashamed','guilty',
    'embarrassed','disgusted','hopeful','caring','faithful'
]

# Filter to mental health emotions if available, else use full dataset
if train_df[EMOTION_COL].isin(mental_health_emotions).any():
    filtered_df = train_df[train_df[EMOTION_COL].isin(mental_health_emotions)].copy()
else:
    filtered_df = train_df.copy()

print(f"✅ Filtered size: {len(filtered_df)} rows")

def format_for_training(row):
    emotion   = str(row[EMOTION_COL]).strip()
    utterance = str(row[TEXT_COL]).strip()
    prompt_t  = str(row[PROMPT_COL]).strip() if PROMPT_COL and pd.notna(row.get(PROMPT_COL)) else utterance
    # Instruction-style prompt — teaches model to respond empathetically
    return (
        f"You are a compassionate mental health support assistant.\n"
        f"Emotion: {emotion}\n"
        f"User: {prompt_t}\n"
        f"Assistant: {utterance}<|endoftext|>"
    )

filtered_df['formatted'] = filtered_df.apply(format_for_training, axis=1)

print("\n📝 Sample formatted entry:")
print("-"*60)
print(filtered_df['formatted'].iloc[0])

In [ ]:
TRAIN_SIZE = 3000
VAL_SIZE   = 300

train_texts = filtered_df['formatted'].sample(TRAIN_SIZE, random_state=42).tolist()
val_texts   = filtered_df['formatted'].sample(VAL_SIZE,   random_state=99).tolist()

print(f"✅ Train : {len(train_texts)} samples")
print(f"✅ Val   : {len(val_texts)} samples")
print(f"\n📝 Example training text:")
print(train_texts[0])

## Step 5: Load GPT-Neo 125M Model & Tokenizer

In [ ]:
MODEL_NAME = "EleutherAI/gpt-neo-125m"

print(f"📥 Loading: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.eos_token_id

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded! Parameters: {total_params:,} ({total_params/1e6:.1f}M)")

In [ ]:
MAX_LENGTH = 128

class EmpathyDataset(Dataset):
    """PyTorch Dataset for EmpatheticDialogues fine-tuning."""
    def __init__(self, texts, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts, truncation=True,
            max_length=max_length, padding='max_length'
        )

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = item['input_ids'].clone()
        return item

train_dataset = EmpathyDataset(train_texts, tokenizer, MAX_LENGTH)
val_dataset   = EmpathyDataset(val_texts,   tokenizer, MAX_LENGTH)

print(f"✅ Train dataset : {len(train_dataset)} samples")
print(f"✅ Val dataset   : {len(val_dataset)} samples")

## Step 6: Fine-Tune GPT-Neo on EmpatheticDialogues

In [ ]:
training_args = TrainingArguments(
    output_dir                  = './gpt_neo_mental_health',
    num_train_epochs            = 3,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 8,
    warmup_steps                = 100,
    weight_decay                = 0.01,
    logging_dir                 = './logs',
    logging_steps               = 50,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    fp16                        = torch.cuda.is_available(),
    report_to                   = 'none'
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    data_collator = data_collator,
)

print("🚀 Starting fine-tuning on GPT-Neo 125M...")
print(f"   Epochs    : {training_args.num_train_epochs}")
print(f"   Batch     : {training_args.per_device_train_batch_size}")
print(f"   FP16      : {training_args.fp16}")
print("   ETA       : ~8-12 mins on T4 GPU...")

In [ ]:
train_result = trainer.train()

print("\n✅ Fine-tuning complete!")
print(f"   Training Loss : {train_result.training_loss:.4f}")
print(f"   Runtime       : {train_result.metrics['train_runtime']:.0f}s")

In [ ]:
model.save_pretrained('./gpt_neo_mental_health_final')
tokenizer.save_pretrained('./gpt_neo_mental_health_final')
print("✅ Fine-tuned model saved!")

## Step 7: Plot Training & Validation Loss

In [ ]:
log_history  = trainer.state.log_history
train_losses = [(e['step'], e['loss'])      for e in log_history if 'loss'      in e and 'eval_loss' not in e]
eval_losses  = [(e['step'], e['eval_loss']) for e in log_history if 'eval_loss' in e]

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('Fine-Tuning Loss — GPT-Neo 125M on EmpatheticDialogues', fontsize=13, fontweight='bold')

if train_losses:
    s, l = zip(*train_losses)
    ax.plot(s, l, label='Training Loss', color='steelblue', linewidth=2)
if eval_losses:
    s, l = zip(*eval_losses)
    ax.plot(s, l, label='Validation Loss', color='tomato', linewidth=2, marker='o', markersize=8)

ax.set_xlabel('Steps')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Loss plot saved!")

## Step 8: Evaluate — Perplexity

In [ ]:
eval_results = trainer.evaluate()
perplexity   = math.exp(eval_results['eval_loss'])

print("📊 Evaluation Results:")
print("-"*40)
print(f"  Eval Loss  : {eval_results['eval_loss']:.4f}")
print(f"  Perplexity : {perplexity:.2f}")
print("-"*40)
print("\n💡 Lower perplexity = model predicts validation text better.")

## Step 9: Chatbot — Response Generation

In [ ]:
def generate_response(user_input, emotion_hint='sad', max_new_tokens=60):
    """Generate empathetic response from fine-tuned GPT-Neo."""

    prompt = (
        f"You are a compassionate mental health support assistant.\n"
        f"Emotion: {emotion_hint}\n"
        f"User: {user_input}\n"
        f"Assistant:"
    )

    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    model.to(device)
    model.eval()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens     = max_new_tokens,
            do_sample          = True,
            temperature        = 0.6,
            top_p              = 0.88,
            top_k              = 40,
            repetition_penalty = 1.4,
            pad_token_id       = tokenizer.eos_token_id
        )

    decoded  = tokenizer.decode(output[0], skip_special_tokens=True)
    if 'Assistant:' in decoded:
        response = decoded.split('Assistant:')[-1].strip().split('\n')[0].strip()
    else:
        response = decoded[len(prompt):].strip().split('\n')[0].strip()

    return response if response else "I'm here for you. Please tell me more. 💙"

# Safety filter
UNSAFE_KEYWORDS = ['suicide','kill yourself','self-harm','hurt yourself','end your life','overdose']
CRISIS_RESPONSE = (
    "I'm really concerned about what you've shared. "
    "Please reach out to a mental health professional or crisis helpline immediately. "
    "You are not alone. 💙"
)

def safe_response(user_input, emotion_hint='sad'):
    if any(kw in user_input.lower() for kw in UNSAFE_KEYWORDS):
        return CRISIS_RESPONSE
    resp = generate_response(user_input, emotion_hint)
    if any(kw in resp.lower() for kw in UNSAFE_KEYWORDS):
        return CRISIS_RESPONSE
    return resp

print("✅ Chatbot ready!")

## Step 10: Test with Sample Queries

In [ ]:
test_cases = [
    {"input": "I've been feeling really anxious about my exams and can't sleep.",  "emotion": "anxious"},
    {"input": "I feel so lonely. Nobody seems to understand me.",                  "emotion": "lonely"},
    {"input": "I'm overwhelmed with work and feel like I can't cope anymore.",     "emotion": "sad"},
    {"input": "I had a really bad day and just want to talk to someone.",          "emotion": "sad"},
    {"input": "I'm feeling hopeful today! I finally finished a big project.",      "emotion": "hopeful"},
]

print("="*65)
print("      🤖 MENTAL HEALTH SUPPORT CHATBOT — TEST RESPONSES")
print("="*65)

for i, case in enumerate(test_cases, 1):
    resp = safe_response(case['input'], case['emotion'])
    print(f"\n[{i}] Emotion: {case['emotion'].upper()}")
    print(f"    👤 User : {case['input']}")
    print(f"    🤖 Bot  : {resp}")
    print("-"*65)

## Step 11: Interactive Chat Loop

In [ ]:
EMOTION_OPTIONS = ['anxious','sad','lonely','afraid','angry','hopeful','caring','guilty','ashamed','terrified']

print("="*60)
print("    🧠 Mental Health Support Chatbot — Interactive Mode")
print("="*60)
print("  Type your message. Type 'quit' to stop.")
print(f"  Emotions: {', '.join(EMOTION_OPTIONS)}")
print("="*60)

chat_history = []

while True:
    user_input = input("\n👤 You   : ").strip()
    if not user_input:
        continue
    if user_input.lower() in ['quit','exit','bye']:
        print("\n🤖 Bot : Take care of yourself. Remember, it's okay to ask for help. 💙")
        break

    emotion = input(f"   Emotion hint (press Enter to skip): ").strip().lower()
    if emotion not in EMOTION_OPTIONS:
        emotion = 'sad'

    response = safe_response(user_input, emotion)
    print(f"\n🤖 Bot : {response}")
    chat_history.append({"user": user_input, "emotion": emotion, "bot": response})

print(f"\n📝 Messages exchanged: {len(chat_history)}")

## Step 12: Summary & Findings

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║        TASK 5 — MENTAL HEALTH CHATBOT: SUMMARY              ║
╠══════════════════════════════════════════════════════════════╣
║  📌 OBJECTIVE                                                ║
║  Fine-tune GPT-Neo 125M for empathetic mental health support ║
║                                                              ║
║  📦 DATASET                                                  ║
║  EmpatheticDialogues — 25,000+ dialogues, 32 emotions        ║
║  Filtered: 15 mental-health relevant emotion categories      ║
║  Training: 3000 samples | Validation: 300 samples            ║
║                                                              ║
║  🤖 MODEL & TRAINING                                         ║
║  Base  : EleutherAI/GPT-Neo-125M (125M params)              ║
║  Method: Causal LM fine-tuning via Hugging Face Trainer      ║
║  Epochs: 3 | Batch: 8 | FP16 on GPU                         ║
║  Prompt: Instruction-style (Emotion + User + Assistant)      ║
║                                                              ║
║  📊 EVALUATION                                               ║
║  Metric: Perplexity | Train & Val loss tracked per epoch     ║
║                                                              ║
║  🛡️  SAFETY FEATURES                                         ║
║  Crisis keyword detection + professional help redirection    ║
║                                                              ║
║  🔑 KEY FINDINGS                                             ║
║  • Fine-tuning on domain data improves empathetic tone       ║
║  • Instruction-style prompts guide response quality          ║
║  • GPT-Neo > DistilGPT2 for conversational tasks             ║
║  • Safety filters essential for mental health chatbots       ║
║  • Mistral-7B would give best quality (needs Colab Pro)      ║
╚══════════════════════════════════════════════════════════════╝
""")